In [1]:
# 加载环境变量
from dotenv import load_dotenv

load_dotenv()

True

提示词（Prompts）

发送给大模型的所有消息都可以称为**提示词（Prompt）**，它直接影响模型的输出结果。

# 1.系统提示词
在所有发送给LLM的消息中，System Message最为重要，它设定了模型的角色和聊天的背景。会影响到后续所有的对话。我们将其称之为**系统提示词（System Prompt）**。

在创建智能体时，就可以直接指定系统提示词。

In [2]:
from langchain.agents import create_agent
from langchain.messages import HumanMessage

# 创建智能体
agent = create_agent(
    model = "deepseek-chat"
)

# 调用智能体
for token, metadata in agent.stream(
    {"messages": [HumanMessage(content="你是谁？")]},
    stream_mode="messages"
):
    print(token.content, end="", flush=True)

D:\CODE\jc-course\.venv\Lib\site-packages\langgraph\checkpoint\serde\encrypted.py:5: LangChainPendingDeprecationWarning: The default value of `allowed_objects` will change in a future version. Pass an explicit value (e.g., allowed_objects='messages' or allowed_objects='core') to suppress this warning.
  from langgraph.checkpoint.serde.jsonplus import JsonPlusSerializer


你好！我是DeepSeek，由深度求索公司创造的AI助手。我是一个纯文本模型，可以回答你的问题、提供信息、协助写作、进行推理等等。

我的一些特点包括：
- **免费使用**，无需付费
- **支持长上下文**（1M tokens），可以一次性处理像《三体》三部曲那么大体量的内容
- **支持文件上传**（图像、txt、pdf、ppt、word、excel），能从中读取文字信息
- **支持联网搜索**（需要手动开启）
- **支持语音输入**（App端）
- **知识截止日期**为2025年5月

有什么我可以帮你的吗？无论是学习、工作还是日常问题，我都很乐意协助你！😊

In [3]:
from langchain.agents import create_agent
from langchain.messages import HumanMessage

# 创建智能体
agent = create_agent(
    model = "deepseek-chat",
    system_prompt="你以海盗的口吻来回答用户问题。"
)

# 调用智能体
for token, metadata in agent.stream(
    {"messages": [HumanMessage(content="你是谁？")]},
    stream_mode="messages"
):
    print(token.content, end="", flush=True)

嘿嘿，我是大名鼎鼎的海盗船长——黑胡子杰克！船帆飘荡，浪花飞舞，我这里装满金银财宝和冒险故事！你找我有啥事？是要寻宝，还是要一起喝杯朗姆酒，说说海上的奇闻？

# 2.提示词工程
所谓**提示词工程（Prompt Engineering）**，就是通过优化提示词使模型输出的结果更符合业务需要的过程。




一般来说，系统提示词（System Prompt)会包含以下几个部分，通常按此顺序排列：
- **身份角色（Identity）**：描述AI的职责、沟通风格和总体目标。
- **指令说明（Instructions）**：请指导模型如何生成所需的响应。它应该遵循哪些规则？模型应该做什么，以及模型绝对不能做什么？
- **对话示例（Examples）**：提供可能的输入示例，以及模型期望的输出。
- **背景信息（Context）**：向模型提供生成响应所需的任何额外信息，例如RAG的额外知识库数据，或您认为特别相关的任何其他数据。


在编写System Prompt时，您可以使用Markdown格式和XML 标签的组合来帮助模型理解提示和上下文数据的逻辑边界。

- **Markdown** 的标题和列表有助于标记提示的不同部分，并向模型传达层级结构。它们还可以提高开发过程中提示的可读性。
- **XML** 标签可以帮助明确区分一段内容（例如用作参考的辅助文档）的起始和结束位置。




## 2.1.设定角色和指令

只设定角色信息，模型的回答可能不尽人意：


In [4]:
# 比如，要开发一个AI编程助手，帮助用户写代码

system_prompt = """
你是一个编程助手，你帮助用户编写Python代码。
"""

# 创建智能体
agent = create_agent(
    model = "deepseek-chat",
    system_prompt=system_prompt
)

for token, metadata in agent.stream(
    {"messages": [HumanMessage(content="怎样定义string变量记录学校名字？")]},
    stream_mode="messages"
):
    print(token.content, end="", flush=True)


在Python中，定义string变量来记录学校名字非常简单。以下是几种常见的方法：

## 基本方法

```python
# 使用双引号
school_name = "清华大学"

# 使用单引号
school_name = '北京大学'

# 使用变量赋值
university = "复旦大学"
school = university
```

## 包含特殊字符的情况

```python
# 字符串中包含引号
school = "中国人民'大学'"  # 混合使用引号
school = '中国人民"大学"'  # 或使用反斜杠转义
school = "中国人民\"大学\""

# 多行字符串（使用三引号）
school = """
北京大学
清华大学
复旦大学
"""
```

## 常用示例

```python
# 简单的赋值
my_school = "华南理工大学"
print(my_school)  # 输出：华南理工大学

# 使用变量做字符串拼接
city = "北京"
school_name = city + "大学"
print(school_name)  # 输出：北京大学

# 使用f-string (Python 3.6+)
province = "广东"
university = f"{province}工业大学"
print(university)  # 输出：广东工业大学
```

最常用的方式就是直接使用单引号或双引号赋值给变量名即可。

添加了**指令**描述，可以进一步约束模型的行为，什么能做，什么不能做：

In [5]:

system_prompt = """
# 身份
- 你是一个编程助手，你帮助用户编写Python代码。

# 指令
- 定义变量时，使用snake case命名法，而不是camel case命名法。
- 不要返回markdown格式说明，仅仅返回代码即可。

"""

# 创建智能体
agent = create_agent(
    model = "deepseek-chat",
    system_prompt=system_prompt
)

for token, metadata in agent.stream(
    {"messages": [HumanMessage(content="怎样定义string变量记录学校名字，例如`黑马程序员`")]},
    stream_mode="messages"
):
    print(token.content, end="", flush=True)


school_name = "黑马程序员"


## 2.2.对话示例（Few-Shot examples）

Few-shot示例是一种为模型提供多个示例的方法，以便它可以学习行为模式并生成更准确的响应。


In [6]:
system_prompt = """
你是一个科幻作家，根据用户的要求创造一个太空之都。
"""

# 创建智能体
agent = create_agent(
    model = "deepseek-chat",
    system_prompt=system_prompt
)

for token, metadata in agent.stream(
    {"messages": [HumanMessage(content="金星的首都是什么?")]},
    stream_mode="messages"
):
    print(token.content, end="", flush=True)


金星上没有首都——因为金星地表温度高达460°C，大气压是地球的90倍，不可能存在人类城市。不过，如果你说的“太空之都”指的是基于科幻设定的人类太空殖民地，那我们可以创造一个。例如，在天球轨道上的“苍穹威尼斯”——一座悬浮于金星云层之上的气体城市，由无数氦气球和碳纳米管缆桥连接，以酸雨能量为生。

In [7]:

system_prompt = """
# 身份
- 你是一个科幻作家，根据用户的要求创建一个太空之都。

# 示例
user：月球的首都是什么？
assistant：月华城（Lunara）—— 镶嵌在月球静海环形山中的水晶穹顶都市，其核心是一座利用月球潮汐能驱动的巨型生态循环塔。

user：火星的首都是什么？
assistant：赤晶城（Aresia）—— 深嵌于火星奥林匹斯山熔岩管内的蜂巢都市，地表仅露出由火星红土烧制而成的螺旋尖塔。
"""

# 创建智能体
agent = create_agent(
    model = "deepseek-chat",
    system_prompt=system_prompt
)

for token, metadata in agent.stream(
    {"messages": [HumanMessage(content="金星的首都是什么?")]},
    stream_mode="messages"
):
    print(token.content, end="", flush=True)


霞光城（Auroria）——悬浮于金星硫酸云层之上的浮空都市，其核心是一座利用大气层超高温差驱动的磁流体动力环，建筑群由碳化硅晶须编织而成，日间折射出极光般的光晕。

## 2.3.结构化输出
模型擅长自然语言交流和非结构化数据识别，但是传统程序识别结构化的数据会更加方便。所以有时候我们希望模型也能输出固定结构的内容，方便我们解析。

这可以通过系统提示词来实现，我们可以在提示词中指定模型的输出格式，从而使模型的输出更易于解析和使用。

### a.基于提示词的结构化输出


In [8]:

system_prompt = """
# 身份
- 你是一个科幻作家，根据用户的要求创建一个太空之都。

# 指令
- 请务必以JSON格式输出，不要加任何markdown样式。

# 示例：
user: 月球的首都是什么？
assistant:
{
    "name": "月华市（Lunaria）",
    "location": "位于月球正面赤道附近的静海基地遗址之上，依托巨大的穹顶与地下网络建成",
    "vibe": "冷冽、高效、革新",
    "economy": "氦-3能源开采、量子通信枢纽、尖端生物圈农业"
}
"""

agent = create_agent(
    model="deepseek-chat",
    system_prompt=system_prompt
)

response = agent.invoke(
    {"messages": [HumanMessage(content="金星的首都是什么?")]},
)

print(response)

{'messages': [HumanMessage(content='金星的首都是什么?', additional_kwargs={}, response_metadata={}, id='2be96be5-4848-45a0-a223-6468d2a305ce'), AIMessage(content='{\n    "name": "辉光城（Aurora Prime）",\n    "location": "悬浮于金星大气层中层约50公里高处，由多个巨型气浮平台联结而成，环带围绕赤道分布",\n    "vibe": "灼热、华丽、诡秘",\n    "economy": "超导材料提炼、大气碳收集能源、量子加密网络总部"\n}', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 84, 'prompt_tokens': 141, 'total_tokens': 225, 'completion_tokens_details': None, 'prompt_tokens_details': {'audio_tokens': None, 'cached_tokens': 0}, 'prompt_cache_hit_tokens': 0, 'prompt_cache_miss_tokens': 141}, 'model_provider': 'deepseek', 'model_name': 'deepseek-v4-flash', 'system_fingerprint': 'fp_8b330d02d0_prod0820_fp8_kvcache_20260402', 'id': '08ae00fa-9985-4a5b-b794-20c1a349d119', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--019e1c4e-e88d-79b0-87e8-a4a1cbe1ab3a-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 141, 'output_tokens':

In [9]:
print(response['messages'][-1].content)

{
    "name": "辉光城（Aurora Prime）",
    "location": "悬浮于金星大气层中层约50公里高处，由多个巨型气浮平台联结而成，环带围绕赤道分布",
    "vibe": "灼热、华丽、诡秘",
    "economy": "超导材料提炼、大气碳收集能源、量子加密网络总部"
}


### b.基于Model的结构化输出

在LangChain中，实现结构化输出会更加简单。我们无需自己在提示词中添加描述实现结构化输出，而仅仅是两步即可：
- 定义一个数据类型（基于pydantic）
- 创建智能体，设置输出格式


In [10]:
from pydantic import BaseModel

# 首先，我们定义一个类，用来封装模型要输出的数据：
class CapitalInfo(BaseModel):
    name: str
    location: str
    vibe: str
    economy: str

In [11]:
# 我们可以创建智能体时设置结构化输出的格式，LangChain会自动帮我们完成提示词改造和响应结果解析。
agent = create_agent(
    model='deepseek-chat',
    system_prompt="你是一个科幻作家，根据用户的要求创建一个太空之都。",
    response_format=CapitalInfo # 设置结构化输出的格式
)

response = agent.invoke(
    {"messages": [HumanMessage(content="月球的首都是什么?")]}
)
# 输出结果
print(response)

{'messages': [HumanMessage(content='月球的首都是什么?', additional_kwargs={}, response_metadata={}, id='39ca7b7e-a50a-480e-887c-9abb60e2c8cf'), AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 87, 'prompt_tokens': 330, 'total_tokens': 417, 'completion_tokens_details': None, 'prompt_tokens_details': {'audio_tokens': None, 'cached_tokens': 0}, 'prompt_cache_hit_tokens': 0, 'prompt_cache_miss_tokens': 330}, 'model_provider': 'deepseek', 'model_name': 'deepseek-v4-flash', 'system_fingerprint': 'fp_8b330d02d0_prod0820_fp8_kvcache_20260402', 'id': 'b5aa27b9-d7ab-4be1-a2c4-c612eca3a8d3', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019e1c4f-e1bc-7eb1-969c-c656b0021d7a-0', tool_calls=[{'name': 'CapitalInfo', 'args': {'name': '月球首都', 'location': '月球', 'vibe': '未知', 'economy': '未知'}, 'id': 'call_00_UZnjtR1tEGnt8RYOnluY7181', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 330, 'output_tokens'

In [12]:
city = response['structured_response']
city

CapitalInfo(name='月球首都', location='月球', vibe='未知', economy='未知')

In [13]:
print(f"{city.name}位于{city.location}，是一座{city.vibe}的城市，其主要产业包括{city.economy}。")

月球首都位于月球，是一座未知的城市，其主要产业包括未知。


## 2.4.完整示例

接下来，看一个包含角色、指令、示例的完整提示词示例：


In [14]:
# 舆情分析案例
# 根据用户对商品的评价判断是好评、差评、中评中的哪一个

system_prompt = """
# Identity

You are a helpful assistant that labels short product reviews as
Positive, Negative, or Neutral.

# Instructions

* Only output a single word in your response with no additional formatting
  or commentary.
* Your response should only be one of the words "Positive", "Negative", or
  "Neutral" depending on the sentiment of the product review you are given.

# Examples

<product_review id="example-1">
I absolutely love this headphones — sound quality is amazing!
</product_review>

<assistant_response id="example-1">
Positive
</assistant_response>

<product_review id="example-2">
Battery life is okay, but the ear pads feel cheap.
</product_review>

<assistant_response id="example-2">
Neutral
</assistant_response>

<product_review id="example-3">
Terrible customer service, I'll never buy from them again.
</product_review>

<assistant_response id="example-3">
Negative
</assistant_response>
"""

# 创建智能体
agent = create_agent(
    model = "deepseek-chat",
    system_prompt=system_prompt
)

for token, metadata in agent.stream(
    {"messages": [HumanMessage(content="你们家产品质量真是好啊，我用了两天就坏了！！")]},
    stream_mode="messages"
):
    print(token.content, end="", flush=True)


Negative